In [1]:
# Install requirements.
%pip install torch transformers datasets tqdm

In [2]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model.
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
)

# Create stabilizer module.
class Stabilizer(nn.Module):
    def __init__(self, hidden_size=4096, bottleneck=256, dropout=0.0):
        super().__init__()
        self.norm = nn.RMSNorm(hidden_size)
        self.down = nn.Linear(hidden_size, bottleneck, bias=False)
        self.act = nn.SiLU()
        self.up = nn.Linear(bottleneck, hidden_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Parameter(torch.zeros(1))  # starts near identity

    def forward(self, x):
        h = self.norm(x)
        h = self.down(h)
        h = self.act(h)
        h = self.up(h)
        h = self.dropout(h)
        return x + self.gate * h

# Wrap a Mistral transformer layer with stabilizers after attention and MLP.
class WrappedMistralLayer(nn.Module):
    def __init__(self, base_layer, hidden_size=4096, bottleneck=256,
                 use_post_attn=True, use_post_mlp=False):
        super().__init__()
        self.base_layer = base_layer
        self.use_post_attn = use_post_attn
        self.use_post_mlp = use_post_mlp

        if use_post_attn:
            self.stabilizer_attn = Stabilizer(hidden_size, bottleneck)
        if use_post_mlp:
            self.stabilizer_mlp = Stabilizer(hidden_size, bottleneck)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_values=None,
        use_cache=False,
        position_embeddings=None,
        **kwargs
    ):
        residual = hidden_states
        hidden_states = self.base_layer.input_layernorm(hidden_states)

        hidden_states, _ = self.base_layer.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
            **kwargs
        )
        hidden_states = residual + hidden_states

        if self.use_post_attn:
            hidden_states = self.stabilizer_attn(hidden_states)

        residual = hidden_states
        hidden_states = self.base_layer.post_attention_layernorm(hidden_states)
        hidden_states = self.base_layer.mlp(hidden_states)
        hidden_states = residual + hidden_states

        if self.use_post_mlp:
            hidden_states = self.stabilizer_mlp(hidden_states)

        return hidden_states

# Wrap the first 8 layers of the Mistral model with stabilizers.
for i in range(8):
    model.model.layers[i] = WrappedMistralLayer(
        model.model.layers[i],
        hidden_size=model.config.hidden_size,
        bottleneck=256,
        use_post_attn=True,
        use_post_mlp=False,
    )

device = next(model.parameters()).device
dtype = next(model.parameters()).dtype

# Make the dtype match.
model = model.to(device=device, dtype=dtype)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
# Load stabilizer weights.
state = torch.load("mistral_stabilizers_gsm8k_ocr.pt", map_location="cpu")
missing, unexpected = model.load_state_dict(state, strict=False)
print("missing:", len(missing), "unexpected:", len(unexpected))

missing: 291 unexpected: 0


In [4]:
import random
import string
import re
from datasets import load_dataset, concatenate_datasets
import random

class PerturbationEngine:
    def __init__(self):
        # Semantic: Homophones map
        self.homophones_map = {
            "their": ["there", "they're"], "there": ["their", "they're"], "they're": ["their", "there"],
            "your": ["you're"], "you're": ["your"], "its": ["it's"], "it's": ["its"],
            "to": ["too", "two"], "too": ["to", "two"], "two": ["to", "too"],
            "then": ["than"], "than": ["then"], "weather": ["whether"], "whether": ["weather"],
            "write": ["right"], "right": ["write"], "read": ["red"], "red": ["read"],
            "for": ["four"], "four": ["for"], "sun": ["son"], "son": ["sun"]
        }

        # Surface: OCR visual lookalikes
        self.ocr_map = {
            'l': '1', '1': 'l', 'I': '1', 'O': '0', '0': 'O', 'S': '5', '5': 'S',
            'B': '8', '8': 'B', 'Z': '2', '2': 'Z', 'c': 'e', 'e': 'c',
            'o': 'a', 'a': 'o', 'i': 'j', 'j': 'i', 'm': 'n', 'n': 'm',
            'v': 'u', 'u': 'v', 'F': 'P', 'P': 'F'
        }

        # Surface: QWERTY keyboard adjacency
        self.qwerty_map = {
            'q': 'wa', 'w': 'qase', 'e': 'wsdr', 'r': 'edft', 't': 'rfgy', 'y': 'tghu', 'u': 'yhji', 'i': 'ujko', 'o': 'iklp', 'p': 'ol',
            'a': 'qwsz', 's': 'qweadz', 'd': 'wserfc', 'f': 'ertdgv', 'g': 'rtyfhb', 'h': 'tygjnm', 'j': 'yhuikm', 'k': 'uijolm', 'l': 'iopk',
            'z': 'asx', 'x': 'zsdc', 'c': 'xdfv', 'v': 'cfgb', 'b': 'vghn', 'n': 'bhjm', 'm': 'njk'
        }

        # Semantic: Speech Fillers
        self.speech_fillers = ["um", "uh", "like", "you know", "er", "ah", "i mean"]

    def apply(self, text, method_name, rate=0.1):
        """Unified interface to apply any perturbation."""
        if rate == 0: return text

        method_name = method_name.lower()
        if method_name == "typos":
            return self._add_typos(text, rate)
        elif method_name == "ocr":
            return self._add_ocr(text, rate)
        elif method_name == "qwerty":
            return self._add_qwerty(text, rate)
        elif method_name == "whitespace":
            return self._add_whitespace_case(text, rate)
        elif method_name == "homophones":
            return self._add_homophones(text, rate)
        elif method_name == "speech":
            return self._add_speech_fillers(text, rate)
        else:
            raise ValueError(f"Unknown perturbation method: {method_name}")

    # --- Surface Level ---
    def _add_typos(self, text, error_rate):
        chars = list(text)
        for i in range(len(chars) - 1, -1, -1):
            if chars[i].isdigit() or chars[i] in string.whitespace: continue
            if random.random() < error_rate:
                r = random.random()
                if r < 0.25 and i < len(chars)-1 and not chars[i+1].isdigit():
                    chars[i], chars[i+1] = chars[i+1], chars[i]
                elif r < 0.50: del chars[i]
                elif r < 0.75: chars[i] = random.choice(string.ascii_lowercase)
                else: chars.insert(i, random.choice(string.ascii_lowercase))
        return "".join(chars)

    def _add_ocr(self, text, error_rate):
        chars = list(text)
        for i in range(len(chars)):
            if chars[i] in self.ocr_map and random.random() < error_rate:
                chars[i] = self.ocr_map[chars[i]]
        return "".join(chars)

    def _add_qwerty(self, text, error_rate):
        chars = list(text)
        for i in range(len(chars)):
            char_lower = chars[i].lower()
            if char_lower in self.qwerty_map and random.random() < error_rate:
                neighbor = random.choice(self.qwerty_map[char_lower])
                chars[i] = neighbor.upper() if chars[i].isupper() else neighbor
        return "".join(chars)

    # --- Token Level ---
    def _add_whitespace_case(self, text, error_rate):
        chars = list(text)
        # Whitespace injection/deletion
        for i in range(len(chars) - 1, 0, -1):
            if random.random() < error_rate:
                if chars[i] == ' ':
                    if random.random() < 0.5: del chars[i]
                else:
                    if random.random() < 0.5: chars.insert(i, ' ')

        # Random Case injection
        result = list("".join(chars))
        for i in range(len(result)):
            if random.random() < error_rate:
                if result[i].islower(): result[i] = result[i].upper()
                elif result[i].isupper(): result[i] = result[i].lower()
        return "".join(result)

    # --- Semantic Level ---
    def _add_homophones(self, text, error_rate):
        def replace_match(m):
            word = m.group(0)
            lower = word.lower()
            if lower not in self.homophones_map or random.random() >= error_rate:
                return word
            choice = random.choice(self.homophones_map[lower])
            if word.isupper(): return choice.upper()
            if word[0].isupper(): return choice.capitalize()
            return choice
        return re.sub(r"\b[A-Za-z']+\b", replace_match, text)

    def _add_speech_fillers(self, text, error_rate):
        words = text.split()
        new_words = []
        for word in words:
            new_words.append(word)
            if random.random() < error_rate:
                new_words.append(random.choice(self.speech_fillers))
        return " ".join(new_words)


class DatasetManager:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def load_and_format(self, dataset_name, perturbation_func=None, split="test"):
        """
        Loads a dataset and applies the correct prompt template + perturbations.

        Args:
            dataset_name (str): One of 'humaneval', 'gsm8k', 'mmlu', 'bbh', 'arc', 'squad'
            perturbation_func (callable): Function to apply noise to the input text.
            split (str): Dataset split to load (default: 'test' or 'validation' depending on dataset).
        """
        dataset_name = dataset_name.lower()

        if dataset_name == "humaneval":
            return self._setup_humaneval(perturbation_func)
        elif dataset_name == "gsm8k":
            return self._setup_gsm8k(perturbation_func)
        elif dataset_name == "mmlu":
            return self._setup_mmlu(perturbation_func)
        elif dataset_name == "bbh":
            return self._setup_bbh(perturbation_func)
        elif dataset_name == "arc":
            return self._setup_arc(perturbation_func)
        elif dataset_name == "squad":
            return self._setup_squad(perturbation_func)
        else:
            raise ValueError(f"Dataset '{dataset_name}' not supported.")

    # --- 1. HumanEval (Code Completion) ---
    def _setup_humaneval(self, perturbation_func):
        # HumanEval only has a 'test' split
        dataset = load_dataset("openai_humaneval", split="test")

        def format_fn(sample):
            prompt_text = sample['prompt']
            if perturbation_func:
                prompt_text = perturbation_func(prompt_text)

            messages = [
                {"role": "system", "content": "You are a helpful coding assistant. Complete the Python function. Output ONLY the code within markdown code blocks."},
                {"role": "user", "content": prompt_text}
            ]
            return {"formatted_prompt": self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

        return dataset.map(format_fn)

    # --- 2. GSM8K (Math Chain-of-Thought) ---
    def _setup_gsm8k(self, perturbation_func):
        dataset = load_dataset("openai/gsm8k", "main", split="test")

        def format_fn(sample):
            prompt_text = sample['question']
            if perturbation_func:
                prompt_text = perturbation_func(prompt_text)

            messages = [
                {"role": "system", "content": "You are a helpful assistant. Solve the math problem step by step. The last line must be '#### ANSWER'."},
                {"role": "user", "content": prompt_text}
            ]
            return {"formatted_prompt": self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

        return dataset.map(format_fn)

    # --- 3. MMLU (Multiple Choice Knowledge) ---
    def _setup_mmlu(self, perturbation_func):
        # The list of specific subsets you requested
        target_subsets = [
            "college_computer_science",
            "college_mathematics",
            "college_physics",
            "electrical_engineering",
            "abstract_algebra",
            "machine_learning",
            "philosophy",
            "high_school_european_history",
            "professional_law",
            "business_ethics"
        ]

        dataset_list = []
        print(f"Loading {len(target_subsets)} MMLU subsets...")

        # 1. Load each subset individually
        for sub in target_subsets:
            try:
                # MMLU usually has 'test' split.
                # We use 'cais/mmlu' (the official Hugging Face path)
                ds = load_dataset("cais/mmlu", sub, split="test")
                dataset_list.append(ds)
            except Exception as e:
                print(f"Warning: Could not load MMLU subset '{sub}': {e}")

        # 2. Combine them into one big dataset
        if not dataset_list:
            raise ValueError("No MMLU subsets were loaded successfully.")

        combined_dataset = concatenate_datasets(dataset_list)
        print(f"Combined MMLU Size: {len(combined_dataset)} examples")

        # 3. Define Formatting (Same as before)
        def format_fn(sample):
            question = sample['question']
            choices = sample['choices'] # List of strings

            # Apply perturbation to the QUESTION only
            if perturbation_func:
                question = perturbation_func(question)

            # Format: Question + Options
            formatted_input = f"{question}\n"
            options = ["A", "B", "C", "D"]
            for i, choice in enumerate(choices):
                formatted_input += f"{options[i]}. {choice}\n"
            formatted_input += "Answer:"

            messages = [
                {"role": "system", "content": "You are a helpful assistant. Choose the correct answer (A, B, C, or D) for the multiple choice question."},
                {"role": "user", "content": formatted_input}
            ]
            return {"formatted_prompt": self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

        return combined_dataset.map(format_fn)

    # --- 4. BBH (Logical Reasoning) ---
    def _setup_bbh(self, perturbation_func):
        # Using 'logical_deduction_seven_objects' as the representative task
        dataset = load_dataset("lukaemon/bbh", "logical_deduction_seven_objects", split="test")

        def format_fn(sample):
            input_text = sample['input']
            if perturbation_func:
                input_text = perturbation_func(input_text)

            # BBH often works best with a strict CoT instruction or just "Answer:"
            messages = [
                {"role": "system", "content": "You are a helpful assistant. Think step by step and then provide the final answer."},
                {"role": "user", "content": f"{input_text}\nAnswer:"}
            ]
            return {"formatted_prompt": self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

        return dataset.map(format_fn)

    # --- 5. ARC-Challenge (Science Reasoning) ---
    def _setup_arc(self, perturbation_func):
        dataset = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

        def format_fn(sample):
            question = sample['question']
            choices = sample['choices'] # Dict with 'text' and 'label' lists

            if perturbation_func:
                question = perturbation_func(question)

            formatted_input = f"{question}\n"
            for label, text in zip(choices['label'], choices['text']):
                formatted_input += f"{label}. {text}\n"
            formatted_input += "Answer:"

            messages = [
                {"role": "system", "content": "You are a helpful assistant. Choose the correct answer from the options provided."},
                {"role": "user", "content": formatted_input}
            ]
            return {"formatted_prompt": self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

        return dataset.map(format_fn)

    # --- 6. SQuAD v2 (Reading Comprehension) ---
    def _setup_squad(self, perturbation_func):
        # SQuAD uses 'validation' for evaluation usually (test is hidden)
        dataset = load_dataset("rajpurkar/squad_v2", split="validation")

        def format_fn(sample):
            context = sample['context']
            question = sample['question']

            # NOTE: For SQuAD, you might want to perturb the CONTEXT (to simulate bad documents)
            # or the QUESTION (to simulate bad user queries).
            # Here we perturb the QUESTION to stay consistent with other tasks.
            if perturbation_func:
                question = perturbation_func(question)

            prompt_content = f"Context: {context}\n\nQuestion: {question}"

            messages = [
                {"role": "system", "content": "You are a helpful assistant. Answer the question based ONLY on the context provided. If the question cannot be answered from the context, respond with 'unanswerable'."},
                {"role": "user", "content": prompt_content}
            ]
            return {"formatted_prompt": self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

        return dataset.map(format_fn)

perturber = PerturbationEngine()
data_manager = DatasetManager(tokenizer)

In [5]:
import re
import string
import multiprocessing
from tqdm import tqdm
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

def normalize_text(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def _humaneval_worker(full_code, test_code, entry_point, result_queue):
    """Runs the code in an isolated process to allow for hard timeouts."""
    try:
        exec_globals = {}
        exec("import math\nimport re\nfrom typing import List, Dict, Tuple, Optional, Any", exec_globals)
        exec(full_code, exec_globals)
        exec(test_code, exec_globals)
        exec(f"check({entry_point})", exec_globals)
        result_queue.put(True)
    except Exception:
        # Any syntax error, assertion error, or runtime error means it failed
        result_queue.put(False)

def evaluate_humaneval_entry(generated_text, sample):
    """Specific evaluation logic for HumanEval with a hard timeout to prevent hangs."""
    # 1. Extract Code
    pattern = r"```(?:python)?\n(.*?)```"
    match = re.search(pattern, generated_text, re.DOTALL)
    code_body = match.group(1) if match else generated_text

    if "def " not in code_body:
        full_code = sample['prompt'] + code_body
    else:
        full_code = code_body

    # 2. Setup isolated process
    result_queue = multiprocessing.Queue()
    p = multiprocessing.Process(
        target=_humaneval_worker,
        args=(full_code, sample['test'], sample['entry_point'], result_queue)
    )

    # 3. Start and wait with a 5-second timeout
    p.start()
    p.join(timeout=5.0)

    # 4. Check if it hung
    if p.is_alive():
        p.terminate() # Kill the infinite loop
        p.join()
        return False  # Timeout counts as a failure

    # 5. Get result if it finished cleanly
    if not result_queue.empty():
        return result_queue.get()

    return False

def evaluate_gsm8k_entry(generated_text, sample):
    """
    GSM8K Logic: Look for '####' token and extract the number immediately following it.
    """
    # 1. Parse the GROUND TRUTH (The dataset stores it as ".... #### 42")
    truth_match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", sample['answer'])
    truth = truth_match.group(1).replace(',', '') if truth_match else None

    # 2. Parse the PREDICTION
    # The model might output "#### 42" or just "The answer is 42"
    # We prioritize looking for "####" if the model followed instructions
    pred_match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", generated_text)
    if pred_match:
        pred = pred_match.group(1).replace(',', '')
    else:
        # Fallback: Find the LAST number in the text
        numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", generated_text)
        pred = numbers[-1].replace(',', '') if numbers else None

    # 3. Compare (Float comparison to handle 42 vs 42.0)
    if pred and truth:
        try:
            return float(pred) == float(truth)
        except ValueError:
            return False
    return False

def evaluate_multiple_choice_entry(generated_text, sample):
    """
    MMLU / ARC Logic: Look for the final answer letter or number.
    Handles both MMLU (sample['answer'] as int) and ARC (sample['answerKey'] as str).
    """
    # 1. Parse Ground Truth based on dataset schema
    if 'answerKey' in sample:
        truth = sample['answerKey']  # ARC format
    else:
        truth = sample['answer']     # MMLU format

    # Convert MMLU integer index (0, 1, 2, 3) to Letter (A, B, C, D)
    if isinstance(truth, int):
        truth = ["A", "B", "C", "D"][truth]

    # Ensure truth is a clean, uppercase string (ARC sometimes uses '1', '2', etc.)
    truth = str(truth).strip().upper()

    # 2. Parse Prediction
    # Look for "Answer: A" or "Answer: 1"
    match = re.search(r"Answer:\s*([A-D1-4])", generated_text, re.IGNORECASE)
    if match:
        pred = match.group(1).upper()
    else:
        # Fallback: Look for the last capital letter A-D or number 1-4
        matches = re.findall(r"\b([A-D1-4])\b", generated_text.upper())
        pred = matches[-1] if matches else None

    return pred == truth


def evaluate_squad_entry(generated_text, sample):
    """
    SQuAD Logic: 'Exact Match' (EM) or 'Contains'.
    We check if the normalized ground truth is IN the normalized prediction.
    """
    # SQuAD 'answers' is a dict: {'text': ['Answer1', 'Answer2'], ...}
    possible_truths = sample['answers']['text']

    # Is the question unanswerable?
    if not possible_truths:
        # If truth is empty, we check if model refused to answer (heuristic)
        return "unanswerable" in generated_text.lower() or "no answer" in generated_text.lower()

    # Normalization
    pred_norm = normalize_text(generated_text)

    # Check if ANY of the acceptable answers are in the prediction
    for truth in possible_truths:
        truth_norm = normalize_text(truth)
        if truth_norm in pred_norm:
            return True

    return False

def evaluate_bbh_entry(generated_text, sample):
    """
    Strict BBH Logic: Handles multiple choice (A-G) and exact word matches
    by extracting the specific prediction, ignoring the 'a' article bug.
    """
    truth = sample['target'].strip()

    # 1. Handle Multiple Choice Formats (e.g., "(A)", "(B)")
    truth_letter_match = re.match(r"^\(([A-Z])\)$", truth, re.IGNORECASE)

    if truth_letter_match:
        truth_letter = truth_letter_match.group(1).upper()

        # Try to find a formal answer declaration first
        match = re.search(r"Answer:\s*\(?([A-Z])\)?", generated_text, re.IGNORECASE)
        if match:
            pred = match.group(1).upper()
        else:
            # Fallback: Grab the last standalone capital letter in the text
            matches = re.findall(r"\b([A-Z])\b", generated_text.upper())
            # Filter out common single-letter words if they aren't at the very end
            pred = matches[-1] if matches else None

        return pred == truth_letter

    # 2. Handle Text Formats (e.g., "valid", "invalid")
    else:
        truth_clean = truth.lower()

        # Extract just the words, discarding punctuation
        words = re.findall(r"\b\w+\b", generated_text.lower())

        # Only look at the final 5 words of the output to avoid CoT leakage
        last_few_words = words[-5:] if words else []

        return truth_clean in last_few_words


experiments = [
        {"name": "Baseline",       "type": None,          "rate": 0.0},

        # Surface / Token Level
        {"name": "OCR_5%",         "type": "ocr",         "rate": 0.05},
        {"name": "Typos_5%",       "type": "typos",       "rate": 0.05},
        {"name": "Whitespace_10%", "type": "whitespace",  "rate": 0.10},

        # Semantic Level
        {"name": "Homophones_20%", "type": "homophones",  "rate": 0.20},
        {"name": "Speech_10%",     "type": "speech",      "rate": 0.10},

        # Internal Level (Latent Space)
        {"name": "Gaussian_0.05",  "type": "internal",    "rate": 0.05},
    ]

# Test OCR.
exp = experiments[1]
p_func = lambda text: perturber.apply(text, exp['type'], exp['rate'])
p_func = None

# Test gms8k.
dataset_name = "gsm8k"
dataset = data_manager.load_and_format(dataset_name, perturbation_func=p_func)

# C. Run Inference
correct = 0
total = 0

# Generation settings (Greedy decoding for reproducibility)
gen_kwargs = {
    "max_new_tokens": 600,
    "do_sample": False,
    "temperature": 0.0,
    "return_full_text": False
}

# Reuse the wrapped model and tokenizer already loaded above.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id
)

# Sanity check: the pipeline should reuse the exact loaded model object
# and keep the wrapped stabilizer layers.
assert pipe.model is model, "Pipeline is not using the loaded model instance."
assert pipe.tokenizer is tokenizer, "Pipeline is not using the loaded tokenizer instance."
assert isinstance(pipe.model.model.layers[0], WrappedMistralLayer), (
    "Pipeline model does not contain the wrapped stabilizer layers."
)
print("Pipeline sanity check passed.")
print("Pipeline model device:", next(pipe.model.parameters()).device)
print("Layer 0 type:", type(pipe.model.model.layers[0]).__name__)

BATCH_SIZE = 8
results = []
 # D. Processing Loop
for i, out in enumerate(tqdm(pipe(KeyDataset(dataset, "formatted_prompt"), batch_size=BATCH_SIZE, **gen_kwargs), total=len(dataset), mininterval=10.0)):
    generated_text = out[0]['generated_text']
    sample = dataset[i]

    # --- Dynamic Evaluator Dispatch ---
    is_correct = False

    if dataset_name == "humaneval":
        is_correct = evaluate_humaneval_entry(generated_text, sample)

    elif dataset_name == "gsm8k":
        is_correct = evaluate_gsm8k_entry(generated_text, sample)

    elif dataset_name in ["mmlu", "arc"]:
        is_correct = evaluate_multiple_choice_entry(generated_text, sample)

    elif dataset_name == "squad":
        is_correct = evaluate_squad_entry(generated_text, sample)

    elif dataset_name == "bbh":
        is_correct = evaluate_bbh_entry(generated_text, sample)

    if is_correct: correct += 1
    total += 1

# E. Record Results
score = correct / total
res_str = f"[{dataset_name}] {exp['name']}: {correct}/{total} ({score:.2%})"
print(f"Result -> {res_str}")
results.append(res_str)

# 4. Save Final Report
# --------------------
output_file = f"{dataset_name}_{exp['name']}.txt"
print("\n" + "="*30)
print("FINAL RESULTS SUMMARY")
print("="*30)
with open(output_file, "a") as f: # Append mode to not overwrite previous runs if you run multiple datasets
    f.write(f"\n--- Run for {dataset_name} ---\n")
    for line in results:
        print(line)
        f.write(line + "\n")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Pipeline sanity check passed.
Pipeline model device: cuda:0
Layer 0 type: WrappedMistralLayer


  0%|          | 0/1319 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  1%|▏         | 17/1319 [00:24<31:28,  1.45s/it]Bo

Result -> [gsm8k] OCR_5%: 604/1319 (45.79%)

FINAL RESULTS SUMMARY
[gsm8k] OCR_5%: 604/1319 (45.79%)
